In [ ]:
import requests
from bs4 import BeautifulSoup
import re
from app.models import MatchInfo
from app.store import HotMatchStore
from app.utils import get_logger
from typing import List, Optional
import os


logger = get_logger(__name__)


def old_request_hot_match_list() -> List[MatchInfo]:
    """过时了"""
    try:
        url = os.getenv("OUHE_HTML_URL")
        response = requests.get(url, proxies={"http": None}, timeout=5)
        html = response.text
        soup = BeautifulSoup(html, "html.parser")

        # 1. 提取比赛主客队
        # 提取所有 span 标签，从热门赛事和热门排行中间的span提取比赛
        # 根据 VS 前后的 span 标签获取主客队信息
        spans = soup.find_all("span")

        start = None
        for i, s in enumerate(spans):
            if s.get_text(strip=True) == "热门赛事":
                start = i
                break

        end = len(spans)
        for i, s in enumerate(spans[start + 1 :], start + 1):
            if s.get_text(strip=True) == "热门排行":
                end = i
                break
        section = spans[start + 1 : end]

        teams = []
        for i, span in enumerate(section):
            if span.get_text(strip=True).upper() == "VS" and 0 < i < len(section) - 1:
                home = section[i - 1].get_text(strip=True)
                away = section[i + 1].get_text(strip=True)
                teams.append((home, away))

        # 2.提取比赛的id
        # html 的超链接规则:
        # + class 中包含 hotmatch-detail-item 字段
        # + href 中包含  /match?id=任意字符&amp;i=任意字符
        # + 经过 soup 解析的正则需要改为: /match?id=任意字符&i=任意字符
        class_pattern = re.compile(r".*\bhotmatch-detail-item\b.*")
        href_pattern = re.compile(r"/match\?id=([^&]+)&i=([^&]+)")

        match_ids = []

        for a_tag in soup.find_all("a", href=True, class_=class_pattern):
            href = a_tag["href"]
            match = href_pattern.search(href)
            if match:
                matchid = match.group(1)
                if matchid not in match_ids:
                    match_ids.append(matchid)

        # 3. 组装比赛信息
        match_list = []
        for match_id, (home, away) in zip(match_ids, teams):
            match_list.append(MatchInfo(home=home, away=away, match_id=match_id))

        logger.info(f"✅ hot match list: {match_list}")

        # 4.保存到缓存
        # store.save_matches(match_list)

        return match_list
    except requests.exceptions.Timeout:
        logger.error("❌ 请求超时")
        return []

In [4]:
import requests
from bs4 import BeautifulSoup
from app.models import MatchInfo
from typing import List
import os
import json
from dotenv import load_dotenv
from app.store import get_match_store
load_dotenv()

def request_hot_match_list() -> List[MatchInfo]:
    url = os.getenv("OUHE_HTML_URL")
    response = requests.get(url, proxies={"http": None}, timeout=5)
    html = response.text
    soup = BeautifulSoup(html, "html.parser")

    script_tag = soup.find("script", id="__NEXT_DATA__", type="application/json")
    json_object = json.loads(script_tag.get_text())
    match_list = json_object["props"]["pageProps"]["data"]["hotMatchList"]
    print(match_list[0])
    matches = [MatchInfo(**m) for m in match_list]
    get_match_store().save_matches(matches)

    return matches

request_hot_match_list()

{'awayTeam': '维多利亚', 'leagueColor': '#996600', 'matchState': 0, 'focus': False, 'viewpointsAsian': [{'amount': 14, 'nowBet': '一球半', 'nowOddsUp': '1.88', 'nowOddsDown': '1.96'}], 'score': '0:0', 'awayLogo': 'https://alphaball-production.oss-cn-beijing.aliyuncs.com/logo/team/20130913231110.png', 'asianOdds': {'nowBet': '一球半', 'handicap': '1.5', 'nowOddsUp': '1.88', 'nowOddsDown': '1.96'}, 'leagueId': 4, 'awayRank': '17', 'homeTeam': '帕尔梅拉斯', 'season': '2025', 'homeTeamId': 422, 'matchId': 1617624, 'matchStateShow': '未开赛', 'matchNoteCount': 28, 'handicap': '-1', 'jczqOdds': {'0': '10.90', '1': '5.45', '3': '1.17'}, 'league': '巴西甲', 'euroOdds': {'0': '10.05', '1': '5.25', '3': '1.28'}, 'matchTime': 1763591400000, 'homeLogo': 'https://alphaball-production.oss-cn-beijing.aliyuncs.com/logo/team/20130916173305.png', 'jcMatchNo': '20251119001', 'minute': "0'", 'homeRank': '2', 'round': '第37轮', 'matchNo': '20251119001', 'liveEvent': [], 'exflag': 0, 'viewpoints': {'rqspfOdds': [{'3': '1.61'}, {'

[MatchInfo(home='帕尔梅拉斯', away='维多利亚', match_id='1617624', league='巴西甲', matchStateShow='未开赛'),
 MatchInfo(home='格雷米奥', away='瓦斯科达伽马', match_id='1617594', league='巴西甲', matchStateShow='未开赛'),
 MatchInfo(home='弗鲁米嫩塞', away='弗拉门戈', match_id='1617593', league='巴西甲', matchStateShow='未开赛'),
 MatchInfo(home='桑托斯', away='米拉索', match_id='1617596', league='巴西甲', matchStateShow='未开赛'),
 MatchInfo(home='彼得堡联', away='斯托克港', match_id='1679014', league='英甲', matchStateShow='未开赛')]